# Animation API on Google Colab

Use a Colab runtime with GPU enabled.

This notebook asks for the repository URL, clones the project, bootstraps `third_party/FasterLivePortrait` when needed, installs the runtime dependencies, downloads checkpoints, starts `realtime_stream_api.py`, and exposes the UI through the Colab port proxy.

In [ ]:
import os

DEFAULT_REPO_BRANCH = "main"
DEFAULT_REPO_DIR = "/content/animation"
DEFAULT_API_PORT = "8010"

repo_url = os.environ.get("ANIMATION_REPO_URL", "").strip()
if not repo_url:
    repo_url = input("Git repository URL: ").strip()
if not repo_url:
    raise ValueError("Repository URL is required.")

repo_branch = os.environ.get("ANIMATION_REPO_BRANCH", DEFAULT_REPO_BRANCH).strip() or DEFAULT_REPO_BRANCH
repo_dir = os.environ.get("ANIMATION_REPO_DIR", DEFAULT_REPO_DIR).strip() or DEFAULT_REPO_DIR
api_port = os.environ.get("ANIMATION_COLAB_API_PORT", DEFAULT_API_PORT).strip() or DEFAULT_API_PORT
api_token = os.environ.get("ANIMATION_COLAB_API_TOKEN", "").strip()
download_checkpoints_raw = os.environ.get("ANIMATION_COLAB_DOWNLOAD_CHECKPOINTS", "1").strip().lower()
download_checkpoints = download_checkpoints_raw not in {"0", "false", "no"}

os.environ["REPO_URL"] = repo_url
os.environ["REPO_BRANCH"] = repo_branch
os.environ["REPO_DIR"] = repo_dir
os.environ["API_PORT"] = api_port
os.environ["API_TOKEN"] = api_token
os.environ["DOWNLOAD_CHECKPOINTS"] = "1" if download_checkpoints else "0"

print(f"Configured repository: {repo_url}@{repo_branch}")
print(f"Repository path: {repo_dir}")
print(f"API port: {api_port}")
print(f"Download checkpoints: {download_checkpoints}")
print("API token configured" if api_token else "API token disabled")


In [ ]:
%%bash
set -euo pipefail

apt-get update
apt-get install -y ffmpeg git

if [ -d "${REPO_DIR}/.git" ]; then
  git -C "${REPO_DIR}" fetch --depth 1 origin "${REPO_BRANCH}"
  git -C "${REPO_DIR}" checkout "${REPO_BRANCH}"
  git -C "${REPO_DIR}" pull --ff-only origin "${REPO_BRANCH}"
else
  git clone --depth 1 --branch "${REPO_BRANCH}" "${REPO_URL}" "${REPO_DIR}"
fi

cd "${REPO_DIR}"
cp -n .env.example .env || true
bash scripts/bootstrap_faster_liveportrait.sh


In [ ]:
%%bash
set -euo pipefail

cd "${REPO_DIR}"

python -m pip install --upgrade pip
python - <<'PY'
from pathlib import Path

source_path = Path("third_party/FasterLivePortrait/requirements.txt")
target_path = Path("/tmp/faster_liveportrait_colab_requirements.txt")
filtered_lines = [
    line
    for line in source_path.read_text(encoding="utf-8").splitlines()
    if line.strip() != "pycuda"
]
target_path.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")
print(target_path)
PY

python -m pip install -r /tmp/faster_liveportrait_colab_requirements.txt
python -m pip install aiortc==1.14.0 fastapi "uvicorn[standard]" python-multipart av onnxruntime "huggingface_hub[cli]"


In [ ]:
%%bash
set -euo pipefail

cd "${REPO_DIR}"

if [ "${DOWNLOAD_CHECKPOINTS}" != "1" ]; then
  echo "Skipping checkpoint download"
  exit 0
fi

huggingface-cli download warmshao/FasterLivePortrait --local-dir third_party/FasterLivePortrait/checkpoints
huggingface-cli download jdh-algo/JoyVASA --local-dir third_party/FasterLivePortrait/checkpoints/JoyVASA
huggingface-cli download TencentGameMate/chinese-hubert-base --local-dir third_party/FasterLivePortrait/checkpoints/chinese-hubert-base


In [ ]:
%%bash
set -euo pipefail

cd "${REPO_DIR}"

if [ -f /tmp/animation_api.pid ] && kill -0 "$(cat /tmp/animation_api.pid)" 2>/dev/null; then
  kill "$(cat /tmp/animation_api.pid)" || true
  sleep 2
fi

export ANIMATION_BACKEND=onnx
export ANIMATION_TRT_RUNTIME=local
export ANIMATION_IDLE_VIDEO=inputs/idlevid.mp4
export ANIMATION_WARMUP_ENABLED=0
export ANIMATION_API_PORT="${API_PORT}"

if [ -n "${API_TOKEN}" ]; then
  export ANIMATION_API_TOKEN="${API_TOKEN}"
fi

nohup python realtime_stream_api.py \
  --host 0.0.0.0 \
  --port "${API_PORT}" \
  --backend onnx \
  --no-warmup \
  > /tmp/animation_api.log 2>&1 &

echo $! > /tmp/animation_api.pid
sleep 20

HEALTHCHECK_URL="http://127.0.0.1:${API_PORT}/api/health"
if [ -n "${API_TOKEN}" ]; then
  curl -fsS -H "Authorization: Bearer ${API_TOKEN}" "${HEALTHCHECK_URL}"
else
  curl -fsS "${HEALTHCHECK_URL}"
fi


In [ ]:
import os
from IPython.display import HTML, display

api_port = int(os.environ["API_PORT"])
api_token = os.environ["API_TOKEN"]
local_health_url = f"http://127.0.0.1:{api_port}/api/health"
local_ui_url = f"http://127.0.0.1:{api_port}/"
if api_token:
    local_ui_url = f"{local_ui_url}?token={api_token}"

proxy_base_url = ""
try:
    from google.colab.output import eval_js
    proxy_base_url = str(eval_js(f"google.colab.kernel.proxyPort({api_port})"))
except Exception as exc:
    print(f"Colab proxy unavailable: {exc}")
    print(f"Local health URL: {local_health_url}")
    print(f"Local UI URL: {local_ui_url}")
else:
    proxy_ui_url = proxy_base_url
    if api_token:
        separator = "&" if "?" in proxy_ui_url else "?"
        proxy_ui_url = f"{proxy_ui_url}{separator}token={api_token}"
    print(f"Proxy URL: {proxy_base_url}")
    print(f"UI URL: {proxy_ui_url}")
    display(HTML(f'<iframe src="{proxy_ui_url}" width="100%" height="900"></iframe>'))

print("Tail logs with: !tail -n 100 /tmp/animation_api.log")
print("Stop the API with: !kill $(cat /tmp/animation_api.pid)")
